ModuleNotFoundError: No module named 'pandas'

In [ ]:
import pandas as pd

df = pd.read_csv("youtube_generalized_raw.csv")

ModuleNotFoundError: No module named 'pandas'

In [1]:
!pip install pandas


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [10]:
import pandas as pd



In [8]:
df.shape



(43236, 10)

In [12]:
sample = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/interim/labeling_comparison_sample.csv")
disagreements = sample[~sample["agree"]]

# focus specifically on the implausible opposite-valence pairs
suspicious_pairs = [("joy", "disgust"), ("disgust", "joy"), ("sadness", "joy"),
                     ("joy", "sadness"), ("surprise", "joy"), ("joy", "surprise")]

for batch_label, single_label in suspicious_pairs:
    subset = disagreements[
        (disagreements["emotion_batch"] == batch_label) &
        (disagreements["emotion_single"] == single_label)
    ]
    print(f"\n=== batch said {batch_label}, single said {single_label} ({len(subset)} cases) ===")
    for _, row in subset.head(5).iterrows():
        print(f"  Text: {row['text_clean'][:150]}")
        print(f"  (batch: {batch_label}, single: {single_label})\n")


=== batch said joy, single said disgust (22 cases) ===
  Text: 1:13 lol anpad gavaar 'vaigyakniko ne' lol
  (batch: joy, single: disgust)

  Text: 2:30 koi tumhe rakhi badhega bhi nhi dekh ke hi lgta hai chor ho 😂😂😂
  (batch: joy, single: disgust)

  Text: I have got sasta version of the same story in my life too.😂 (Vellepanti ki bhi hadd hai like seriously fake id !!!, tum toh already fake ho (relatives
  (batch: joy, single: disgust)

  Text: Modi ji you have a large heart to beg for forgiveness in front of the nation for failing to implement such wonderful farm laws even though only a sect
  (batch: joy, single: disgust)

  Text: Bhai me gaya tha aaj tak studio ye sale sb fake dikhate hai 😂😂😂😂
  (batch: joy, single: disgust)


=== batch said disgust, single said joy (9 cases) ===
  Text: Mortien se to machhar bhi nahi marte mei kya khaak marungi 😂😂😭❤️
  (batch: disgust, single: joy)

  Text: MISHRA,  NARAYAN, UPADHAYAY,  TIWARI  PATHAK  pandit ji ra.di nachawe laglan
  (batch: disg

In [1]:
import json
from collections import Counter

# Load the JSON file
with open("/Users/harshaggarwal/Projects_4/hinemo_project/data/interim/hinglish_teaser_5k.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Count emotions
emotion_counts = Counter(item["emotion"] for item in data)

# Print results
print(emotion_counts)

Counter({'Neutral': 2611, 'Happy': 1064, 'Curious': 583, 'Frustrated': 186, 'Sad': 179, 'Humor': 140, 'Fear': 90, 'Surprised': 71, 'Angry': 62, 'Disgusted': 14})


In [2]:
file_path = "/Users/harshaggarwal/Projects_4/hinemo_project/data/Extra(fortesting)/hinglish_train (1).txt"

count = 0

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("meta"):
            count += 1

print("Number of entries:", count)

Number of entries: 15133


In [3]:


with open(file_path, "r", encoding="utf-8") as f:
    content = f.read().strip()

entries = [e.strip() for e in content.split("\n\n") if e.strip()]

seen = {}
duplicates = []

for i, entry in enumerate(entries):
    lines = entry.split("\n")

    # Ignore the meta line
    tokens = [line.split("\t")[0] for line in lines[1:] if "\t" in line]
    tweet = " ".join(tokens)

    if tweet in seen:
        duplicates.append((seen[tweet], i, tweet))
    else:
        seen[tweet] = i

print(f"Total entries: {len(entries)}")
print(f"Duplicate tweets: {len(duplicates)}")

# Show first 10 duplicates
for first, second, tweet in duplicates[:10]:
    print(f"\nDuplicate found:")
    print(f"Entry {first} and Entry {second}")
    print(tweet)

Total entries: 15131
Duplicate tweets: 0


In [4]:

ids = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("meta"):
            parts = line.strip().split("\t")
            ids.append(parts[1])

from collections import Counter

counts = Counter(ids)
duplicate_ids = {k: v for k, v in counts.items() if v > 1}

print(f"Total IDs: {len(ids)}")
print(f"Unique IDs: {len(counts)}")
print(f"Duplicate IDs: {len(duplicate_ids)}")

if duplicate_ids:
    print("Duplicate IDs:", duplicate_ids)

Total IDs: 15133
Unique IDs: 15132
Duplicate IDs: 1
Duplicate IDs: {'Eng': 2}


In [5]:
import csv
import random
import string

# Input and output files
input_file = "/Users/harshaggarwal/Projects_4/hinemo_project/data/Extra(fortesting)/hinglish_train (1).txt"
output_file = "/Users/harshaggarwal/Projects_4/hinemo_project/data/raw/twitter_takenfromsemevalpaper.csv"

# Function to generate unique 10-character IDs
used_ids = set()

def generate_id():
    while True:
        uid = ''.join(random.choices(string.ascii_letters + string.digits, k=10))
        if uid not in used_ids:
            used_ids.add(uid)
            return uid

# Read the file
with open(input_file, "r", encoding="utf-8") as f:
    content = f.read().strip()

entries = [e.strip() for e in content.split("\n\n") if e.strip()]

rows = []

for entry in entries:
    lines = entry.split("\n")

    # Skip malformed entries
    if not lines or not lines[0].startswith("meta"):
        continue

    tokens = []

    # Ignore the meta line
    for line in lines[1:]:
        parts = line.split("\t")
        if len(parts) >= 2:
            tokens.append(parts[0])

    text_clean = " ".join(tokens)

    rows.append({
        "source_id": generate_id(),
        "source": "twitter",
        "text_clean": text_clean
    })

# Save to CSV
with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["source_id", "source", "text_clean"]
    )
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved {len(rows)} tweets to '{output_file}'.")

Saved 15131 tweets to '/Users/harshaggarwal/Projects_4/hinemo_project/data/raw/twitter_takenfromsemevalpaper.csv'.


In [10]:
import pandas as pd
import re

# Load the CSV
csv_file = "/Users/harshaggarwal/Projects_4/hinemo_project/data/raw/twitter_takenfromsemevalpaper.csv"
df = pd.read_csv(csv_file)

import re

def clean_text(text):
    text = str(text)

    # Remove @ mentions whether tokenized or not
    text = re.sub(r'@\s*[A-Za-z0-9_]+', '', text)

    # Remove hashtags whether tokenized or not
    text = re.sub(r'#\s*[A-Za-z0-9_]+', '', text)

    # Remove normal URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove tokenized URLs like:
    # https // t co / abc123
    # https // t . co / abc123
    text = re.sub(
        r'https?\s*/?\s*/\s*t\s*\.?\s*co\s*/\s*\S+',
        '',
        text,
        flags=re.IGNORECASE
    )

    # Remove any remaining http/https tokens
    text = re.sub(r'\bhttps?\b', '', text, flags=re.IGNORECASE)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Add the new column
df["cleaned_text"] = df["text"].apply(clean_text)

# Save the updated CSV
df.to_csv(csv_file, index=False, encoding="utf-8")

print("Added 'cleaned_text' column successfully.")

Added 'cleaned_text' column successfully.


In [14]:
import pandas as pd
df = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/processed/sentimix_batch_labeled(15131).csv")
print(f"Total rows: {len(df)}")
print(df["emotion"].value_counts())

Total rows: 15131
emotion
joy         4318
anger       4043
neutral     3691
disgust     1897
sadness      662
fear         411
surprise     109
Name: count, dtype: int64


In [2]:
import json
import pandas as pd

with open("../data/Extra(fortesting)/hinglish_teaser_5k.json") as f:
    data = json.load(f)
df_5k = pd.DataFrame(data)

sample = df_5k.sample(20, random_state=42)
for _, row in sample.iterrows():
    print(f"[{row['emotion']}] {row['text']}")
    print()

[Neutral] Bhai please gift kar do na

[Neutral] Non living wala me samajh gaya tha Lekin comments ab karani padh rahi he

[Neutral] Sister app sir per dupatta Liya karo

[Curious] Like seriously Agar aise hi khatam karna tha to 24 hrs ka liya hi kyun?

[Curious] Vacancy kitni aane wali hai?

[Neutral] Good looking aaj Kal log bhote hai yaar pehle toh koi nahi bolta tha Deep words MANOJ BAJPAPYEE SAHAB

[Happy] NICE VLOGS BHAI I LOVE IT PLEASE HAMESHA DAL NA

[Neutral] Tu kyu dusre ka video use kr rha

[Happy] Sir Mary Kom 6 times world champion hai 5 nahi But Great Efforts Amazing Video

[Frustrated] Kunal karma ka satkar ho rha hai ya roast kar rahe ho Itna boring roast episode

[Happy] Bahut acchi baat hai batane ke liye dhanyvad

[Neutral] Jise computer ka knowledge nahi hai kya vo bhi kar sakte hai

[Neutral] Bohot sahi Thumbnail change karo please keep it up

[Surprised] Bhai update to do kya hua fir mare ke nahi

[Happy] Woow bohot shanti ka abhas ho raha hai sir thank you thank 

In [7]:
import pandas as pd

youtube = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/processed/youtube_labeled(43236).csv")
sentimix = pd.read_csv("../data/processed/sentimix_batch_labeled(15131).csv")
teaser5k = pd.read_csv("../data/processed/teaser5k_labeled(5000).csv")

for name, df in [("youtube", youtube), ("sentimix", sentimix), ("teaser5k", teaser5k)]:
    print(f"=== {name} ===")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print()

=== youtube ===
Shape: (26708, 11)
Columns: ['source_id', 'source', 'video_id', 'target_emotion', 'text', 'like_count', 'published_at', '_source_file', 'text_clean', 'language_bucket', 'emotion']

=== sentimix ===
Shape: (15131, 5)
Columns: ['source_id', 'source', 'text', 'cleaned_text', 'emotion']

=== teaser5k ===
Shape: (5000, 11)
Columns: ['id', 'text', 'intent', 'emotion', 'toxicity', 'sarcasm', 'language', 'quality_score', 'label_confidence', 'label_method', 'is_short']



In [8]:
for name, df in [("youtube", youtube), ("sentimix", sentimix), ("teaser5k", teaser5k)]:
    print(f"=== {name} emotion distribution ===")
    counts = df["emotion"].value_counts()
    pcts = df["emotion"].value_counts(normalize=True) * 100
    summary = pd.DataFrame({"count": counts, "pct": pcts.round(1)})
    print(summary)
    print()

=== youtube emotion distribution ===
          count   pct
emotion              
joy        9568  35.8
anger      5531  20.7
disgust    5201  19.5
sadness    3907  14.6
fear       1608   6.0
surprise    893   3.3

=== sentimix emotion distribution ===
          count   pct
emotion              
joy        4318  28.5
anger      4043  26.7
neutral    3691  24.4
disgust    1897  12.5
sadness     662   4.4
fear        411   2.7
surprise    109   0.7

=== teaser5k emotion distribution ===
          count   pct
emotion              
neutral    2603  52.1
joy        1242  24.8
anger       519  10.4
disgust     234   4.7
fear        188   3.8
sadness     182   3.6
surprise     32   0.6



In [9]:
unwanted = youtube[youtube["emotion"].isin(["unknown", "error", "neutral"])]
print(f"YouTube rows with unwanted labels (unknown/error/neutral): {len(unwanted)}")
print(youtube["emotion"].value_counts())

YouTube rows with unwanted labels (unknown/error/neutral): 0
emotion
joy         9568
anger       5531
disgust     5201
sadness     3907
fear        1608
surprise     893
Name: count, dtype: int64


In [10]:
id_cols = {"youtube": "source_id", "sentimix": "source_id", "teaser5k": "id"}

for name, df in [("youtube", youtube), ("sentimix", sentimix), ("teaser5k", teaser5k)]:
    col = id_cols[name]
    dupes = df[col].duplicated().sum()
    print(f"{name}: {dupes} duplicate {col} values out of {len(df)}")

youtube: 0 duplicate source_id values out of 26708
sentimix: 0 duplicate source_id values out of 15131
teaser5k: 0 duplicate id values out of 5000


In [11]:
text_cols = {"youtube": "text_clean", "sentimix": "cleaned_text", "teaser5k": "text"}

for name, df in [("youtube", youtube), ("sentimix", sentimix), ("teaser5k", teaser5k)]:
    col = text_cols[name]
    dupes = df[col].duplicated().sum()
    print(f"{name}: {dupes} duplicate {col} values out of {len(df)}")

youtube: 132 duplicate text_clean values out of 26708
sentimix: 175 duplicate cleaned_text values out of 15131
teaser5k: 0 duplicate text values out of 5000


In [12]:
yt_text = set(youtube["text_clean"].str.strip().str.lower())
sm_text = set(sentimix["cleaned_text"].str.strip().str.lower())
t5k_text = set(teaser5k["text"].str.strip().str.lower())

print(f"YouTube ∩ SentiMix: {len(yt_text & sm_text)}")
print(f"YouTube ∩ teaser5k: {len(yt_text & t5k_text)}")
print(f"SentiMix ∩ teaser5k: {len(sm_text & t5k_text)}")

YouTube ∩ SentiMix: 0
YouTube ∩ teaser5k: 0
SentiMix ∩ teaser5k: 0


In [13]:
before_yt = len(youtube)
youtube = youtube.drop_duplicates(subset=["text_clean"], keep="first")
print(f"youtube: {before_yt} -> {len(youtube)} (-{before_yt - len(youtube)})")

before_sm = len(sentimix)
sentimix = sentimix.drop_duplicates(subset=["cleaned_text"], keep="first")
print(f"sentimix: {before_sm} -> {len(sentimix)} (-{before_sm - len(sentimix)})")

before_t5k = len(teaser5k)
teaser5k = teaser5k.drop_duplicates(subset=["text"], keep="first")
print(f"teaser5k: {before_t5k} -> {len(teaser5k)} (no duplicates found earlier, expect no change)")

youtube: 26708 -> 26576 (-132)
sentimix: 15131 -> 14956 (-175)
teaser5k: 5000 -> 5000 (no duplicates found earlier, expect no change)


In [14]:
youtube.to_csv("../data/processed/youtube_labeled_dedup(26708).csv", index=False)
sentimix.to_csv("../data/processed/sentimix_labeled_dedup(14956).csv", index=False)
teaser5k.to_csv("../data/processed/teaser5k_labeled_dedup(5000).csv", index=False)

print("Saved deduplicated versions:")
print(f"  youtube_labeled_dedup.csv: {len(youtube)} rows")
print(f"  sentimix_labeled_dedup.csv: {len(sentimix)} rows")
print(f"  teaser5k_labeled_dedup.csv: {len(teaser5k)} rows")

Saved deduplicated versions:
  youtube_labeled_dedup.csv: 26576 rows
  sentimix_labeled_dedup.csv: 14956 rows
  teaser5k_labeled_dedup.csv: 5000 rows


In [18]:
import pandas as pd

round2 = pd.read_csv("../data/interim/round2_labeled_stage4(13231).csv")

counts = round2["emotion_label"].value_counts()
pcts = round2["emotion_label"].value_counts(normalize=True) * 100
summary = pd.DataFrame({"count": counts, "pct": pcts.round(1)})

print(f"Total: {len(round2)}\n")
print(summary)

Total: 13231

               count   pct
emotion_label             
neutral         3200  24.2
joy             2798  21.1
sadness         2779  21.0
anger           2365  17.9
disgust         1447  10.9
fear             534   4.0
surprise         107   0.8


In [19]:
import pandas as pd

round2 = pd.read_csv("../data/interim/round2_labeled_stage4(13231).csv")
print(f"round2 before dedup: {len(round2)}")

before = len(round2)
round2 = round2.drop_duplicates(subset=["source_id"])
print(f"round2 after source_id dedup: {len(round2)} (-{before - len(round2)})")

before = len(round2)
round2 = round2.drop_duplicates(subset=["text_clean"])
print(f"round2 after text_clean dedup: {len(round2)} (-{before - len(round2)})")

round2 before dedup: 13231
round2 after source_id dedup: 13231 (-0)
round2 after text_clean dedup: 12861 (-370)


In [21]:
# reload the other three (deduplicated versions from before)
youtube = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/processed/youtube_labeled_removed_duplicates(26708).csv")
sentimix = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/processed/sentimix_labeled_removed_duplicates(14956).csv")
teaser5k = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/processed/teaser5k_labeled_removed_duplicates(5000).csv")

yt_text = set(youtube["text_clean"].str.strip().str.lower())
sm_text = set(sentimix["cleaned_text"].str.strip().str.lower())
t5k_text = set(teaser5k["text"].str.strip().str.lower())
r2_text = set(round2["text_clean"].str.strip().str.lower())

print(f"round2 ∩ youtube: {len(r2_text & yt_text)}")
print(f"round2 ∩ sentimix: {len(r2_text & sm_text)}")
print(f"round2 ∩ teaser5k: {len(r2_text & t5k_text)}")

round2 ∩ youtube: 10
round2 ∩ sentimix: 0
round2 ∩ teaser5k: 0


In [22]:
def standardize(df, id_col, text_col, emotion_col, source_name):
    out = pd.DataFrame({
        "id": df[id_col].astype(str),
        "text": df[text_col],
        "emotion": df[emotion_col],
        "source": source_name,
    })
    return out

yt_std = standardize(youtube, "source_id", "text_clean", "emotion", "youtube_round1")
sm_std = standardize(sentimix, "source_id", "cleaned_text", "emotion", "sentimix")
t5k_std = standardize(teaser5k, "id", "text", "emotion", "teaser5k")
r2_std = standardize(round2, "source_id", "text_clean", "emotion_label", "youtube_round2")

TARGET_EMOTIONS = {"anger", "joy", "sadness", "disgust"}

merged = pd.concat([yt_std, sm_std, t5k_std, r2_std], ignore_index=True)
before = len(merged)
merged = merged[merged["emotion"].isin(TARGET_EMOTIONS)]
print(f"Before emotion filter: {before}")
print(f"After keeping only {TARGET_EMOTIONS}: {len(merged)}")

before = len(merged)
merged = merged.drop_duplicates(subset=["text"])
print(f"After final cross-source text dedup: {len(merged)} (-{before - len(merged)})")

print("\nFinal distribution:")
print(merged["emotion"].value_counts())

Before emotion filter: 59393
After keeping only {'sadness', 'anger', 'disgust', 'joy'}: 46180
After final cross-source text dedup: 46174 (-6)

Final distribution:
emotion
joy        17715
anger      12369
disgust     8698
sadness     7392
Name: count, dtype: int64


In [23]:
merged.to_csv("../data/processed/hinemo_dataset_draft_1.csv", index=False)
print(f"Saved {len(merged)} rows -> hinemo_dataset_draft_1.csv")

Saved 46174 rows -> hinemo_dataset_draft_1.csv


In [24]:
sample = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/processed/hinemo_dataset_draft_1.csv")

In [25]:
sample.head()

,id,text,emotion,source
0,UgwHBaZa3bpEnAncy3h4AaABAg,इतिहास याद रखेगा की हजारों गिधड पत्रकारों के ब...,joy,youtube_round1
1,Ugx6lFR6QtDEUHfXbON4AaABAg,Sachai ko Sacchai Batana Bhi Aaj ke time me ba...,joy,youtube_round1
2,UgxVhB_B75japh1ZbaN4AaABAg,Sir.. Aaj apne hamare किसान भाइयों par video b...,joy,youtube_round1
3,Ugzd5Vg4druXrie6Kd94AaABAg,Thanks bro चुनाव se phle जनता ko जगाने के लिए 🎉🎉🎉,joy,youtube_round1
4,UgzNxTBjse78ZSqbqqd4AaABAg,😢me from farmers family.....thnku dhruv sir fo...,sadness,youtube_round1


In [33]:
sample[sample["source"] == ""teaser5k""].head()

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2016376234.py, line 1)

In [37]:
merged[merged["source"] == "teaser5k"].head()

,id,text,emotion,source
34880,1,Ravi ka humour best hai usko vulgarity ki need nahi padti joke funny banane ke liye Although main Samay ka hater nahi hu par mujhe Ravi type ki comedy better lagti,joy,teaser5k
34881,3,"Op sunny, 1no. Nice mast",joy,teaser5k
34882,4,robotic culture se job lose hongi job ni hoga to jarurte puri ni hongi har aadmi khud ko utna devolope ni kr skta jitna chahiye so jarurte pura krne ke liye sb chori wagaira wagaira krenge isliye robotic culture so bad,disgust,teaser5k
34883,5,O bhai Not good bro bekar hai sach me,sadness,teaser5k
34884,6,Bohot hi bdhiya,joy,teaser5k


In [36]:
merged = pd.read_csv("../data/processed/hinemo_dataset_draft_1.csv")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

print(f"Total rows: {len(merged)}, columns: {merged.columns.tolist()}\n")

for source_name in merged["source"].unique():
    print(f"\n{'='*60}")
    print(f"SOURCE: {source_name}")
    print('='*60)
    display(merged[merged["source"] == source_name].sample(5, random_state=42))

Total rows: 46174, columns: ['id', 'text', 'emotion', 'source']


SOURCE: youtube_round1


,id,text,emotion,source
22048,UgzhBc4YMubiV7xlfsh4AaABAg,Tum bohot furtila larki ho,joy,youtube_round1
7524,UgwshGxso0x1n-GrKvd4AaABAg,Bhai tatti mein a raha hai,disgust,youtube_round1
4424,UgwPnBuTdqgDMi8V3Kx4AaABAg,I justtt loveseeeeeeeeee her acting 🤣❤❤❤❤❤,joy,youtube_round1
3268,Ugx5OmzdigSh8SrgHJ14AaABAg,Wow hamre yaha to sutali boom fodne se padosiyo ki fatt ke haath me aa jati hai😂😅,anger,youtube_round1
10984,UgxleXfDNx6iw_wDEhJ4AaABAg,Govt. Is sleeping and so public 😢😢,sadness,youtube_round1



SOURCE: sentimix


,id,text,emotion,source
25095,pZsYV6d7BK,_ 🙄🙄🙄 nahi 😂😂😂 by the way apki love …,disgust,sentimix
27607,bOtM1GOEV4,RT Like u always say that “ kuch b ho sakta hai ” I was singing kuch b 😅🙊 my self made language 🙊🙈 but thank u so much for c …,joy,sentimix
28089,AdstM3LAGS,Bkwas krwalo tm logonnsy bs .. Khud kartoot anay ki nahi thi . I spit on your grave,anger,sentimix
32116,AStCliRoFe,I LOVE THEM SO MUCH THANK YOU MGA ATE KO MUAAH,joy,sentimix
30237,p1CPjsdcGd,mileya mileya - jigar sariya priya andrews and rekha bharadwaj ( a god tier song I'm so in love with this ),joy,sentimix



SOURCE: teaser5k


,id,text,emotion,source
35162,681,Kya baat hai Collaboration kar lo dosto,joy,teaser5k
35359,1113,Bhai phle to bahut hasate to but at end yrr emotional kr dete ho emotional hi nhi mtlb rula dete ho bhai,sadness,teaser5k
36187,3067,Aap toh legend ho Kya haal hai sir? ek number,joy,teaser5k
36134,2942,Jane anjane me hi sahi lekin ye bnda hamare culture history ko galt sabit krne me hi lga rehta h.,disgust,teaser5k
35387,1178,Really Seriously can't believe k abhi tak nahi dekhi pls watch Bohot hi achhi movie hai sabko dekhni chahiye,joy,teaser5k



SOURCE: youtube_round2


,id,text,emotion,source
39590,Ugy1XE6qaPBMg5pUIyZ4AaABAg,Sir RATAN TATA ke death ke baad yeh video dekh raha hun 😢💪🏻❤️,sadness,youtube_round2
39542,UgwNnQSMKD1I_JHiQ4d4AaABAg,"Jo bhi is duniya me aya he ek din jarur jayega , yahi jeevan ka niyam lekin kya karke jayega yahi ose great banati he - RIP RATAN TATA SIR ♥️❤",sadness,youtube_round2
44837,UgyRnhRYiB3JUinD1QR4AaABAg,Desh ki janta pagal h,anger,youtube_round2
43893,UgyYlDyb-oxDCQWwgm94AaABAg,Nitin Gadkari ji ko sob na Development ka Guru samajta tha par Nitin Gadkari ji to Guru ghantal adwami nikla 😂😂😂,disgust,youtube_round2
42469,Ugysd36D9Evg7edML1x4AaABAg,Mtlb bhai tu na saaf saaf kyu ni bolta h terko paise milte h indian current government ko nicha dikhane ke liye,anger,youtube_round2


In [38]:
merged[merged["source"] == "teaser5k"].head()

,id,text,emotion,source
34880,1,Ravi ka humour best hai usko vulgarity ki need nahi padti joke funny banane ke liye Although main Samay ka hater nahi hu par mujhe Ravi type ki comedy better lagti,joy,teaser5k
34881,3,"Op sunny, 1no. Nice mast",joy,teaser5k
34882,4,robotic culture se job lose hongi job ni hoga to jarurte puri ni hongi har aadmi khud ko utna devolope ni kr skta jitna chahiye so jarurte pura krne ke liye sb chori wagaira wagaira krenge isliye robotic culture so bad,disgust,teaser5k
34883,5,O bhai Not good bro bekar hai sach me,sadness,teaser5k
34884,6,Bohot hi bdhiya,joy,teaser5k


In [ ]:
before_yt = len(youtube)
youtube = youtube.drop_duplicates(subset=["text_clean"], keep="first")
print(f"youtube: {before_yt} -> {len(youtube)} (-{before_yt - len(youtube)})")

before_sm = len(sentimix)
sentimix = sentimix.drop_duplicates(subset=["cleaned_text"], keep="first")
print(f"sentimix: {before_sm} -> {len(sentimix)} (-{before_sm - len(sentimix)})")

before_t5k = len(teaser5k)
teaser5k = teaser5k.drop_duplicates(subset=["text"], keep="first")
print(f"teaser5k: {before_t5k} -> {len(teaser5k)} (no duplicates found earlier, expect no change)")

youtube: 26708 -> 26576 (-132)
sentimix: 15131 -> 14956 (-175)
teaser5k: 5000 -> 5000 (no duplicates found earlier, expect no change)


In [39]:
merged = pd.read_csv("../data/processed/hinemo_dataset_draft_1.csv")
print(merged["emotion"].value_counts())
print(merged["emotion"].value_counts(normalize=True) * 100)

emotion
joy        17715
anger      12369
disgust     8698
sadness     7392
Name: count, dtype: int64
emotion
joy        38.365747
anger      26.787803
disgust    18.837441
sadness    16.009009
Name: proportion, dtype: float64


In [40]:
print(pd.crosstab(merged["source"], merged["emotion"]))

emotion         anger  disgust   joy  sadness
source                                       
sentimix         4003     1878  4263      651
teaser5k          519      234  1242      182
youtube_round1   5517     5161  9545     3862
youtube_round2   2330     1425  2665     2697


In [41]:
merged["token_count"] = merged["text"].str.split().str.len()
print(merged.groupby("emotion")["token_count"].describe())

           count       mean        std  min   25%   50%   75%     max
emotion                                                              
anger    12369.0  23.167596  29.444262  3.0  11.0  18.0  24.0   912.0
disgust   8698.0  20.647850  21.331568  2.0  10.0  16.0  24.0   611.0
joy      17715.0  15.272199  18.075825  2.0   7.0  11.0  19.0   902.0
sadness   7392.0  17.575352  26.904548  2.0   6.0  11.0  20.0  1124.0


In [42]:
print(f"Exact duplicates: {merged['text'].duplicated().sum()}")

Exact duplicates: 0


In [43]:
merged.sample(30, random_state=1)[["text", "emotion", "source"]]

,text,emotion,source
21653,Maut aajayee par aisaa din na aaye 😢,sadness,youtube_round1
42923,Indian gadaro ko goli maro ..behan ke lovde hai sab congressi,anger,youtube_round2
46161,Koi baat nhi sab sahi hai 😊\nBolo\nJai Sree Ram 😊,joy,youtube_round2
21890,Yrr it's feels like it's my own story and your parents is so mine parents 😭😭😭\nEspecially your dad is my mom☝🏼😭😭😭😭,joy,youtube_round1
4819,Aap or neetu di bhut achi lagte hai shat me,joy,youtube_round1
15276,Ache din aa gy bhai,joy,youtube_round1
30622,RT __ phrp Happy birthday _ jeb ! Enjoy your day mama base loves you 😊,joy,sentimix
21988,WE NEED MORE STORY TIMESSSSSSSSSSSSSSSSSSSSSSSSSSS,joy,youtube_round1
19987,bhai mujhe tho tera walk in wardrobe laajawaab laga,joy,youtube_round1
21262,15:28 lavade lagna had h time😂😂,disgust,youtube_round1


In [44]:
merged.nlargest(5, "token_count")[["text", "emotion", "source", "token_count"]]

text  \
43014  The Pulwama attack has left all of us in a state-of-shock, sorrow and the raging outbursts are seen across the country. Here’s Grandmaster Shifuji Shaurya Bhardwaj lashing out fire and seeking answers to questions which have gone unanswered!\n\nOn 14 February 2019, the nation witnessed the worst ever militant attack in over 2 decades which has left us everyone in a state of shock and anger. A convoy of 78 vehicles transporting more than 2,500 Central Reserve Police Force (CRPF) personnel from Jammu to Srinagar was traveling on National Highway 44. The convoy had left Jammu around 3:32 IST. Usually, it is a convoy of 1000 soldiers, however, on this day a large number of personnel’s were deputed in the vehicles as two days prior to the heinous Pulwama attack the highway was shut down.\n\npulwama attack 14th feb 2019\n\nThis deadliest attack witnessed in 2 decades of Kashmir’s insurgency has hit the ‘NO TOLERANCE’ cord of every Indian. There is no denial about respecting the Indian constitution and the fundamentals it has laid down. The feeling right now has crossed the borders of rage or exasperation and I feel that the ones playing such hideous crimes should be ripped apart in between. It’s high time, we all wake up as true citizens of India and give a vehement reply to this heinous act of terrorism.\nTo-date, we have been only talking and condemning about any unfortunate incidents that occur. And to everyone’s dismay, the aftermath is forgotten as if it was a Valentine celebration. No objection towards these celebrations and one should certainly celebrate happiness. But remember, there is no bigger valentine than our country and the patriotism fever cannot rise only on 15th August and 26th January!!\nLet’s get back to the incident and try finding answers to some questions.\nWe lost 40 CRPF jawans and many more continue to battle for their lives. A convoy of 2500 CRPF personnel’s were commuting from Jammu to Srinagar. They were about to get deputed for the safety of Kashmiris’ when a Scorpio loaded with more than 350 kilograms of explosives and IED rammed into one of the buses — carrying 35-40 soldiers, and the next is a tragedy we all will never forget!\nWhat are we doing about it? Everyone spoke a lot about it, condemned the act, and paid tribute to the 40 martyr Jawans who lost their life. But we still are far away from the real concern! The incident took place at Awantipura, which is a sector in Pulwama district, and which is considered to be the safest route for the army. How did the Scorpio reach there? What sought of breach happened internally, so that the car could find an entry in a restricted area?\nThe biggest threat to us today is that we are still avoiding from finding the root cause of this incident. Surprisingly nobody wants to talk about it today? We get to hear a lot about how we shall be addressing and replying back to this situation. But this is not a situation that has come abruptly! It is a well-planned conspiracy and no one is interested in finding the whereabouts and how it paved its way all to the safest root of the army! Why???\nUmpteen times I have shared videos to help the Kashmiris’ understand the gravity of terrorist attacks and not get influenced by them. Under the name of religion, the Islamis and Jihadis are spreading terror and think they can get away with this impotency?? How can any religion think and act in an extremely non-humanitarian way? Killing innocent people, when it is not even a war-zone reflects nothing but their impotency!\nA couple of other questions which has gone under the covers still remain a mystery.\nFrom where did they get access to IED, and how can a country which is begging for economic and all kinds of varied supports to sustain, bear the cost of 350 kgs explosives? Who leaked the itinerary of 2500 soldiers traveling towards Srinagar? How did this Scorpio go unnoticed? Without any internal betrayer, this conspiracy would have been impossible!!\nWorldwide, 

In [47]:
import pandas as pd

# Load dataset
merged = pd.read_csv("../data/processed/hinemo_dataset_draft_1.csv")

TOTAL_SAMPLE = 1000
RANDOM_STATE = 2026

# Number of rows in each source
source_counts = merged["source"].value_counts()

# Initial proportional allocation
sample_sizes = (source_counts / len(merged) * TOTAL_SAMPLE).round().astype(int)

# Ensure the total is exactly 1000
difference = TOTAL_SAMPLE - sample_sizes.sum()
if difference != 0:
    largest_source = sample_sizes.idxmax()
    sample_sizes[largest_source] += difference

print("Sample allocation:")
print(pd.DataFrame({
    "Dataset Size": source_counts,
    "Sample Size": sample_sizes
}))
print(f"\nTotal sample = {sample_sizes.sum()}")

# Draw the samples
samples = []

for source, n in sample_sizes.items():
    source_df = merged[merged["source"] == source]
    sampled = source_df.sample(n=n, random_state=RANDOM_STATE)
    samples.append(sampled)

# Combine and shuffle
final_sample = (
    pd.concat(samples)
      .sample(frac=1, random_state=RANDOM_STATE)
      .reset_index(drop=True)
)

# Save
final_sample.to_csv(
    "../data/manual_relabeling_sample_for_kohens_kappa_1000.csv",
    index=False
)

print("Saved to ../data/manual_relabeling_sample_for_kohens_kappa_1000.csv")

Sample allocation:
                Dataset Size  Sample Size
source                                   
youtube_round1         24085          522
sentimix               10795          234
youtube_round2          9117          197
teaser5k                2177           47

Total sample = 1000
Saved to ../data/manual_relabeling_sample_for_kohens_kappa_1000.csv


In [ ]:
import pandas as pd

# Load dataset
merged = pd.read_csv("../data/processed/hinemo_dataset_draft_1.csv")

TOTAL_SAMPLE = 1000
RANDOM_STATE = 2026

samples = []

# Number of samples to draw from each source
source_counts = merged["source"].value_counts()
source_sample_sizes = (
    (source_counts / len(merged) * TOTAL_SAMPLE)
    .round()
    .astype(int)
)

# Make sure total is exactly TOTAL_SAMPLE
difference = TOTAL_SAMPLE - source_sample_sizes.sum()
if difference != 0:
    source_sample_sizes[source_sample_sizes.idxmax()] += difference

allocation = []

for source, source_n in source_sample_sizes.items():

    source_df = merged[merged["source"] == source]

    # Emotion distribution within this source
    emotion_counts = source_df["emotion"].value_counts()

    emotion_sample_sizes = (
        (emotion_counts / len(source_df) * source_n)
        .round()
        .astype(int)
    )

    # Ensure total matches source_n
    diff = source_n - emotion_sample_sizes.sum()
    if diff != 0:
        emotion_sample_sizes[emotion_sample_sizes.idxmax()] += diff

    for emotion, emotion_n in emotion_sample_sizes.items():

        emotion_df = source_df[source_df["emotion"] == emotion]

        sampled = emotion_df.sample(
            n=min(emotion_n, len(emotion_df)),
            random_state=RANDOM_STATE
        )

        samples.append(sampled)

        allocation.append({
            "Source": source,
            "Emotion": emotion,
            "Dataset Size": len(emotion_df),
            "Sample Size": len(sampled)
        })

# Combine and shuffle
final_sample = (
    pd.concat(samples)
      .sample(frac=1, random_state=RANDOM_STATE)
      .reset_index(drop=True)
)

# Save files
final_sample.to_csv(
    "../data/manual_relabeling_sample_for_kohens_kappa_1000.csv",
    index=False
)

allocation_df = pd.DataFrame(allocation)
allocation_df.to_csv(
    "../data/manual_relabeling_sample_allocation.csv",
    index=False
)

print(f"Final sample size: {len(final_sample)}")
print("\nAllocation:")
print(allocation_df)

Final sample size: 1000

Allocation:
            Source  Emotion  Dataset Size  Sample Size
0   youtube_round1      joy          9545          206
1   youtube_round1    anger          5517          120
2   youtube_round1  disgust          5161          112
3   youtube_round1  sadness          3862           84
4         sentimix      joy          4263           92
5         sentimix    anger          4003           87
6         sentimix  disgust          1878           41
7         sentimix  sadness           651           14
8   youtube_round2  sadness          2697           58
9   youtube_round2      joy          2665           58
10  youtube_round2    anger          2330           50
11  youtube_round2  disgust          1425           31
12        teaser5k      joy          1242           27
13        teaser5k    anger           519           11
14        teaser5k  disgust           234            5
15        teaser5k  sadness           182            4


In [50]:
import pandas as pd

In [52]:
df = pd.read_csv("../data/processed/hinemo_dataset_draft_1.csv")

In [54]:
df.count()

id             46174
text           46174
gpt_emotion    46174
source         46174
dtype: int64

In [56]:
df["gpt_emotion"].value_counts()

gpt_emotion
joy        17715
anger      12369
disgust     8698
sadness     7392
Name: count, dtype: int64

In [63]:
import pandas as pd

merged = pd.read_csv("../data/processed/hinemo_dataset_draft_1(46174).csv")

SAMPLE_PATH = "../data/manual_relabeling_sample_for_kohens_kappa_1000.csv"
ALLOCATION_PATH = "../data/manual_relabeling_sample_allocation.csv"

existing = pd.read_csv(SAMPLE_PATH)

MIN_PER_SOURCE = 150
RANDOM_STATE = 2026

topups = []
allocation = []

for source, group in merged.groupby("source"):
    already_have = len(existing[existing["source"] == source])
    need = max(0, MIN_PER_SOURCE - already_have)

    if need == 0:
        continue

    # candidates not already in the existing sample
    pool = group[~group["id"].isin(existing["id"])]

    # stratify the top-up by emotion, same logic as the original script
    emotion_counts = pool["gpt_emotion"].value_counts()
    emotion_sizes = (emotion_counts / len(pool) * need).round().astype(int)
    diff = need - emotion_sizes.sum()
    if diff != 0:
        emotion_sizes[emotion_sizes.idxmax()] += diff

    for emotion, n in emotion_sizes.items():
        emotion_pool = pool[pool["gpt_emotion"] == emotion]
        take = min(n, len(emotion_pool))
        sampled = emotion_pool.sample(n=take, random_state=RANDOM_STATE)
        topups.append(sampled)

        allocation.append({
            "Source": source,
            "Emotion": emotion,
            "Dataset Size": len(emotion_pool),
            "Sample Size": take
        })

if topups:
    topup_df = pd.concat(topups).reset_index(drop=True)
    final_sample = pd.concat([existing, topup_df]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

    # overwrite the original sample file in place
    final_sample.to_csv(SAMPLE_PATH, index=False)

    # append the new allocation rows to the existing allocation file
    old_allocation = pd.read_csv(ALLOCATION_PATH)
    new_allocation = pd.concat([old_allocation, pd.DataFrame(allocation)], ignore_index=True)
    new_allocation.to_csv(ALLOCATION_PATH, index=False)

    print(f"Original: {len(existing)}, Added: {len(topup_df)}, Final: {len(final_sample)}")
else:
    print("No top-up needed — all sources already at or above the minimum.")

Original: 1000, Added: 103, Final: 1103


In [3]:
df = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/manual_relabeling_sample_for_kohens_kappa_1000.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/harshaggarwal/Projects_4/hinemo_project/data/manual_relabeling_sample_for_kohens_kappa_1000.csv'

In [68]:
df[id].value_counts()

KeyError: 4720049232

In [2]:
import pandas as pd

# Load files
yt2 = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/processed/round2_labeled_stage4(13231).csv")
sentimix = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/processed/sentimix_labeled_removed_duplicates(14956).csv")
teaser = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/processed/teaser5k_labeled_removed_duplicates(5000).csv")
yt1 = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/processed/youtube_labeled_removed_duplicates(26708).csv")

# Emotion distribution
print("yt2")
print(yt2["gpt_emotion"].value_counts())
print()

print("Sentimix")
print(sentimix["gpt_emotion"].value_counts())
print()

print("Teaser5k")
print(teaser["gpt_emotion"].value_counts())
print()

print("yt1")
print(yt1["gpt_emotion"].value_counts())

yt2
gpt_emotion
neutral     3200
joy         2798
sadness     2779
anger       2365
disgust     1447
fear         534
surprise     107
Name: count, dtype: int64

Sentimix
gpt_emotion
joy         4263
anger       4003
neutral     3644
disgust     1878
sadness      651
fear         408
surprise     109
Name: count, dtype: int64

Teaser5k
gpt_emotion
neutral     2603
joy         1242
anger        519
disgust      234
fear         188
sadness      182
surprise      32
Name: count, dtype: int64

yt1
gpt_emotion
joy         9545
anger       5517
disgust     5161
sadness     3862
fear        1601
surprise     890
Name: count, dtype: int64


In [1]:
import pandas as pd

In [ ]:
kappa_df = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/manual_relabeling_samples/Final Manually Relabelled.csv")
print(f"Shape: {kappa_df.shape}")
print(f"Columns: {kappa_df.columns.tolist()}")
print(f"\nManual label distribution:")
print(kappa_df["manual emotion(joy,sadness,anger disgust)"].value_counts())
print(f"\nMissing manual labels: {kappa_df['manual emotion(joy,sadness,anger disgust)'].isna().sum()}")
print(f"\nSample rows:")
kappa_df.head(10)


Shape: (1103, 4)
Columns: ['id', 'text', 'manual emotion(joy,sadness,anger disgust)', 'source']

Manual label distribution:
manual emotion(joy,sadness,anger disgust)
joy        464
anger      333
sadness    177
disgust    117
Name: count, dtype: int64

Missing manual labels: 12

Sample rows:


,id,text,"manual emotion(joy,sadness,anger disgust)",source
0,UgzESsoKlbq6D1lCVSp4AaABAg,ðŸ˜‚ðŸ˜‚ðŸ˜‚ðŸ˜‚ðŸ˜‚bichra koi filam dakh ka a...,sadness,youtube_round2
1,UgzkoaxQRyyfTUuh_SN4AaABAg,I love you maa â¤â¤â¤â¤â¤â¤â¤,joy,youtube_round1
2,XNcLcPAcyi,Naib tehsildar Ka paper leak ho gaya Hajaro me...,anger,sentimix
3,snc3sxtv7k,M . K B . C tim jaise kamino ko saza kab hogi ...,anger,sentimix
4,Ugyl9-2w5YAsdA37X-J4AaABAg,Bhai ambani se bada ghar kharid liye but swimi...,sadness,youtube_round1
5,4062,Suraj bhai ke Jo Dil Se fan Hai subscriber lik...,joy,teaser5k
6,UgwgOzhozEA-ylXDtq94AaABAg,Mujhe lagta hai ki tum India ke liye video nah...,disgust,youtube_round2
7,UgzZeJTkJHVvGGB0NLp4AaABAg,Fir bhi logo ko chahiye modi modi modi wah re ...,anger,youtube_round1
8,UgycT98G-LTMKvUoaz94AaABAg,School college mein kahan se phone allowed hai...,anger,youtube_round1
9,UgzWXNvqkPoy496F67N4AaABAg,Saare youtuber se best Ghar Aapka h....bahut h...,joy,youtube_round1


In [10]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

# Load manual labels
manual = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/manual_relabeling_samples/Final Manually Relabelled.csv", 
                     encoding="utf-8")
manual = manual.rename(columns={"manual emotion(joy,sadness,anger disgust)": "manual_label"})
manual["id"] = manual["id"].astype(str)


# Load merged dataset to get GPT labels
merged = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/processed/hinemo_dataset_draft_1(46174).csv")
merged["id"] = merged["id"].astype(str)

# Join on id
combined = manual.merge(merged[["id", "gpt_emotion"]], on="id", how="inner")
combined = combined.rename(columns={"gpt_emotion": "gpt_label"})

print(f"Manual labels: {len(manual)}")
print(f"Matched rows: {len(combined)}")
print(f"Unmatched: {len(manual) - len(combined)}")
print(f"\nManual label distribution:")
print(combined["manual_label"].value_counts())
print(f"\nGPT label distribution (same rows):")
print(combined["gpt_label"].value_counts())

Manual labels: 1103
Matched rows: 1103
Unmatched: 0

Manual label distribution:
manual_label
joy        464
anger      333
sadness    177
disgust    117
Name: count, dtype: int64

GPT label distribution (same rows):
gpt_label
joy        441
anger      293
disgust    200
sadness    169
Name: count, dtype: int64


In [11]:
# Check for nulls/non-string values
print("Manual label nulls:", combined["manual_label"].isna().sum())
print("GPT label nulls:", combined["gpt_label"].isna().sum())
print("Manual label unique:", combined["manual_label"].unique())
print("GPT label unique:", combined["gpt_label"].unique())

Manual label nulls: 12
GPT label nulls: 0
Manual label unique: <StringArray>
['sadness', 'joy', 'anger', 'disgust', nan]
Length: 5, dtype: str
GPT label unique: <StringArray>
['disgust', 'joy', 'anger', 'sadness']
Length: 4, dtype: str


In [12]:
# Drop rows where manual label is missing
combined_clean = combined.dropna(subset=["manual_label"])
print(f"Rows after dropping 12 unannotated: {len(combined_clean)}")

# Overall kappa
overall_kappa = cohen_kappa_score(combined_clean["manual_label"], combined_clean["gpt_label"])
print(f"\nOverall Cohen's Kappa: {overall_kappa:.4f}")

if overall_kappa >= 0.8:
    interp = "Almost perfect agreement"
elif overall_kappa >= 0.6:
    interp = "Substantial agreement"
elif overall_kappa >= 0.4:
    interp = "Moderate agreement"
elif overall_kappa >= 0.2:
    interp = "Fair agreement"
else:
    interp = "Poor agreement — investigate"
print(f"Interpretation: {interp}")

# Per-emotion kappa
print(f"\nPer-emotion Kappa:")
for emotion in ["joy", "anger", "sadness", "disgust"]:
    binary_manual = (combined_clean["manual_label"] == emotion).astype(int)
    binary_gpt = (combined_clean["gpt_label"] == emotion).astype(int)
    k = cohen_kappa_score(binary_manual, binary_gpt)
    print(f"  {emotion}: {k:.4f}")

# Confusion matrix
print(f"\nConfusion matrix (rows=manual, cols=GPT):")
labels = ["anger", "disgust", "joy", "sadness"]
cm = confusion_matrix(combined_clean["manual_label"], combined_clean["gpt_label"], labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df)

# Raw agreement rate
agreement = (combined_clean["manual_label"] == combined_clean["gpt_label"]).mean()
print(f"\nRaw agreement rate: {agreement:.1%}")

# Disagreement breakdown
disagreements = combined_clean[combined_clean["manual_label"] != combined_clean["gpt_label"]]
print(f"Total disagreements: {len(disagreements)} ({len(disagreements)/len(combined_clean):.1%})")
print(f"\nDisagreement pairs (manual -> GPT):")
print(disagreements.groupby(["manual_label", "gpt_label"]).size().sort_values(ascending=False))

Rows after dropping 12 unannotated: 1091

Overall Cohen's Kappa: 0.5704
Interpretation: Moderate agreement

Per-emotion Kappa:
  joy: 0.8016
  anger: 0.5026
  sadness: 0.5192
  disgust: 0.2836

Confusion matrix (rows=manual, cols=GPT):


NameError: name 'confusion_matrix' is not defined

In [7]:
def categorize_disagreement(manual, gpt):
    pair = (manual, gpt)
    if pair in [("anger", "disgust"), ("disgust", "anger")]:
        return "anger-disgust boundary (adjacent emotions, theoretically expected)"
    elif pair in [("sadness", "anger"), ("anger", "sadness"),
                  ("sadness", "disgust"), ("disgust", "sadness")]:
        return "negative-emotion boundary (valence-adjacent)"
    elif "joy" in pair:
        return "positive-negative crossing (sarcasm/mocking misfire)"
    else:
        return "other"

disagreements = combined_clean[combined_clean["manual_label"] != combined_clean["gpt_label"]].copy()
disagreements["error_category"] = disagreements.apply(
    lambda r: categorize_disagreement(r["manual_label"], r["gpt_label"]), axis=1
)

print(disagreements["error_category"].value_counts())
print(f"\n% of all disagreements:")
print(disagreements["error_category"].value_counts(normalize=True) * 100)

NameError: name 'combined_clean' is not defined

In [6]:
def categorize_disagreement(manual, gpt):
    pair = (manual, gpt)
    if pair in [("anger", "disgust"), ("disgust", "anger")]:
        return "anger-disgust boundary (adjacent emotions, theoretically expected)"
    elif pair in [("sadness", "anger"), ("anger", "sadness"),
                  ("sadness", "disgust"), ("disgust", "sadness")]:
        return "negative-emotion boundary (valence-adjacent)"
    elif "joy" in pair:
        return "positive-negative crossing (sarcasm/mocking misfire)"
    else:
        return "other"

disagreements = combined_clean[combined_clean["manual_label"] != combined_clean["gpt_label"]].copy()
disagreements["error_category"] = disagreements.apply(
    lambda r: categorize_disagreement(r["manual_label"], r["gpt_label"]), axis=1
)

print(disagreements["error_category"].value_counts())
print(f"\n% of all disagreements:")
print(disagreements["error_category"].value_counts(normalize=True) * 100)

NameError: name 'combined_clean' is not defined

In [ ]:
corpus = """
khaunga, jinki, cheej, raaz, maulana, mut, angrej, mutra, milavat, khilwad, dadagiri,
karr, jani, legi, sune, karle, shabd, adhikar, moodi, baaj, kartay, neech, choudhary,
sadhu, chutiyo, khudko, hmne, guzar, lutere, abki, jantar, chutiyapa, sali, rishtedaar,
mujhy, bhadve, watan, karay, kuttey, anshan, kahne, eske, hamse, nach, jeetna, karnay,
hamein, kase, bdiya, jabardasti, bahana, utni, mza, bolege, etni, khalistani, sikho,
aandolan, safal, bheek, itihas, murti, saram, sarm, peda, azaad, sudhrenge, desho,
jitane, nicha, chaar, matalab, karana, baaz, yesa, theka, dalte, dikhawa, lagegi,
dadaji, viswas, konsi, hamary, muja, nashe, baj, jaanta, afgani, aankhon, kabja, kabza,
aaja, dipawali, doston, diwas, pana, lutne, niyam, mandiron, bachana, nich, sunai,
dadu, krle, kru, aache, zindgi, dikhaye, tarh, karane, bahiya, jimmedari, rkhe, isase,
sunder, chalna, sabji, pyaj, daam, darte, jhalak, aaega, tang, chize, aas, jayga,
awaam, adhikari, sidhe, huya, kisika, giya, sutta, bharne, jau, deshbhakti, ayegi,
khayal, takat, karam, aakh, thori, vivek, sache, rakshak, patwari, rupay, dein, looto,
khtam, chaat, khayega, khake, jel, kharidne, rupya, prasad, fokat, daala, netaon,
purav, chahate, bindu, marwa, patrakar, dimak, buddhe, gundo, fukra, khatra, yahe,
namaz, chootiya, gher, macchar, dafa, abhinandan, sabha, bharosha, ganna, jabardast,
bula, mahnat, niti, namah, kurta, humse, haare, sikhna, baithi, kamjor, jaiso, rojgar,
uppar, kahate, kardi, ldke, banke, ankh, jhandu, patidar, waalon, waalo, tay, lagna,
chood, jija, jhuk, kabse, billi, mahaan, chowkidaar, bhejna, nikl, jase, jhoota, wjh,
harr, sachme, imaan, pgl, lijiye, ganga, saste, mangne, aakho, sote, dhrm, marungi,
hogai, aapni, zyda, padhi, nikli, charitra, gam, sikhne, hojaye, aagyi, dakh, ekdm,
bachche, ped, maahi, drd, kitana, fati, nikalne, buddhi, essa, aara, akeli, mre, ghusa,
shiv, mukt, yese, sabut, leya, ang, hathi, dege, neend, chahiya, kisine, pap, doshi,
unlogo, fasa, dikhe, saabit, kerne, jag, padhna, araha, krwaya, kamaya, sbb, hamra,
daalo, rishtedar, rhegi, samman, ilaj, nagar, nikali, majboor, padai, puchna, biki,
manenge, sawaal, kum, ajkal, ganne, gandagi, milenge, dusare, aanand, zabardast, giri,
keede, skty, chalaya, bandar, kyaaa, duba, sasti, kamina, adda, assalam, unme, mulla,
ladna, gaandu, seekh, thha, musalmaan, peti, banwaya, aisy, baray, gadbadi, pehly,
machar, shareef, khandan, pakistaniyo, aay, aawaz, kayi, tamasha, kimat, haalat, banae,
asliyat, puchta, aagye, toda, tez, bazi, krdi, joo, phale, bheem, btate, chahata,
sabne, dange, jumla, kahana, failane, gumrah, thali, wal, jina, todne, videshi, kaan,
hataya, khilaya, karre, dekhlo, pare, chalao, laao, khujli, yuvraj, jitoge, kahega,
sharab, maara, tmhari, tusi, karwao, dhanya, huwe, kero, gaddari, kaal, manege, wajha,
hotha, aaram, ekbar, bhayya, karty, chatni, gol, bhavishya, sapno, kharida, dalaal,
vidhayak, karachi, chadda, halal, kutton, tussi, liberals, bund, footage, choot,
pahuch, fayada, kucha, tahe, kalam, bhonk, kaunsa, garibon, bhagavan, leni, midiya,
uspar, bankar, paat, babe, sanskari, dhoti, bhagwan, gurudwara, chadhava, pandit,
sindoor, iftar, roza, namaz, kanjar, launda, chamcho, mahavir, dhongi, tarki, jhandu,
patrkar, chhetri, gumne, kota, fouji, chahre, badhta, divya, anjaam, kerti, sadma, kie, atul, thana, aisha,
bati, samjhna, sas, sansar, volg, bivi, dekhker, inlog, kea, khatir, sahiba, budhi, pisach,
insab, aabe, vaale, gudda, masa, buhut, mahanga, tamatar, sathiya, darne, hil, bhat, hassi,
gharke, pichli, dekhle, tujhko, koy, bakvas, daalte, lootane, khai, khod, laash, putra,
kholna, dekta, gea, sunkr, ekdin, chahega, ajata, vishnu, pairon, krwao, faku, mansikta,
param, dand, hatyara, samaaj, islami, chaal, balatkari, jhuta, ehi, pocha, hanumanji, soye,
juthi, khooni, pra, achchhi, flim, bapa, sajish, sattam, jant, enka, bhu, khulega, bujurg,
chir, jayenga, manane, siyar, koee, bolay, premi, kamini, apaka, nyaay, banay, po, taraha,
um, ahe, eya, sall, gus, pero, padhana, krlu, bhaari, mahenat, badhi, andaz, bnke, padhane,
akhiri, mra, seena, phati, badai, nokar, kholta, chune, samajhti, kliye, ruko, mehanat, ela,
zayada, tatha, gatiya, mag, shuruwat, garm, napunsak, chadao, tode, esme, bisleri, insaano,
dinn, shauk, mager, padhti, duri, padhaya, gram, mlas, koshis, ajib, krwate, krwane, hve,
kardenge, gyanesh, aadarniya, nta, pelta, rokne, nyi, pagalo, gappe, mazaaa, talash, khayi,
badhane, mon, karon, bikul, iman, itani, milakar, mitron, kanhi, hoh, saktha, madarsa,
barbaadi, haatho, thhe, murdo, ghada, chijon, chhupane, benchod, rozgar, tanashahi, muddon,
sachchai, peena, piyo, aagay, dharne, ayese, guss, muda, fasai, sade, oho, chadh, becha,
chaddha, kejru, waaah, kaala, ghaat, alu, aparichit, bant, bhuka, parivahan, paan, sae,
dawat, samosa, gandgi, aaegi, ghtiya, baarish, karsakte, jhaat, lgata, itti, rahege,
karobar, abadi, gadiya, takke, lgra, daud, lout, kadva, uthayenge, schi, mzaa, ispar, eski,
shaha, ghutno, vipin, chodte, nakami, tadipar, bnya, ankit, bahaya, chupane, idr, marwaya,
fansa, aah, apn, por, leek, osko, sati, badhega, bnte, tumahare, hyn, batayi, lakhon, nak,
kuc, dalwa, safayi, adhar, agyi, parth, choty, chijo, bhartiyon, papu, karwata, bolre,
enki, shikshak, andhera, yaaa, pu, urs, uthayi, karvaya, khoya, kahen, pakhandiyo, paidal,
bhindi, hawas, chamdi, bki, nye, isl, kalakar, hasa, lath, enn, ehh, behtarin, tagdi,
humhare, dhande, bhkt, sevak, pheli, jariya, gulaam, chodd, betiyo, hojae, vishesh, hilana,
mitra, katega, yu, istemaal, pahen, milunga, khudki, moose, chalenge, piya, kut, shaq,
chaahe, harkato, hmmmm, lege, haraam, kriye, paisey, rakhega, pla, gorakhpur, santo, sourab,
ts, tattu, bulate, thekedaar, chadhana, kasoor, hindhu, khudka, mannat, pataa, haque, hoto,
chors, hazaaron, chadane, chaukidaar, vishwash, paresh, mazaak, chokidaar, babari,
bewakoofi, dharmantaran, roko, aamir, takle, kaate, chadhane, sabkuch, churane, yuhi, jh,
chudi, hindo, toti, kamse, laxman, mubark, isliya, mathura, dh, jugad, jgah, lute, baksha,
badrinath, karnewale, haqeeqat, manav, shraap, bahiskar, dhup, irada, pehchan, swadeshi,
moja, gaddaron, thug, chodne, khte, samajwadi, wesa, hamza, kahun, aum, gusa, bazar,
patrakaar, kasmir, aak, chalak, karliya, riste, paesa, quraan, pbuh, panchayat, sakt, mehar,
aurton, ghata, hutiya, krn, bord, pichhle, andhere, jesse, hathon, panda, sanghi, parde,
agayi, saurab, dhakke, mistri, dhabba, rahu, jagan, dhoop, sabr, juta, kyuu, parsad, daant,
smajh, doosron, dekhati, lagy, qasam, hogay, aajtk, likee, yap, agey, rhege, krungi, vd,
bndi, myyy, insaneee, muj, thisss, ut, vjah, js, doooo, plis, puraa, storyyy, ajtak,
padhke, bho, pdi, mail, pahan, behenchod, apoorvaaaa, pados, insaneeee, raita, makkar,
wardi, gareebon, budha, bhutni, joota, fekte, bhoke, bl, indo, wesy, namastey, sakoon,
jany, khopdi, buddhu, unhon, rehmt, dhol, leny, hama, kahika, merey, pichwada, ble, krain,
jaahil, rajneta, angrezi, jummah, bandy, len, jhaadu, lein, phenk, jaisy, ramdan, lya,
gaaon, iz, jatey, hmlogo, likho, illaj, bies, pedaish, piyari, sakenge, jnu, walekum,
uthai, pun, chan, badhte, daag, kuti, patrakaron, sehri, daikh, girvi, kutay, chhavi, akki,
samji, zrur, ora, bulati, inshallha, saa, boondi, jindawad, muqabla, bhtt, hinsa, yei,
jesay, tasveer, wasiyo, hazaar, lain, zeal, kudiye, chipke, jawans, dhasu, jashn, chaplusi,
zabardasti, haraami, sans, chahne, mga, lassi, kub, ato, bachey, padhare, isteefa, rum,
balke, jish, batne, khanzeer, isay, labh, gando, misss, sood, aatankwaad, bassi, ayya,
lutera, arbo, goribke, sifuji, paendabad, mite, piliz, malamaal, cororo, chhal, choona,
satik, morcha, jispe, kissano, jaban, boltay, banav, toofan, wakeup, dharuv, chidiya,
milaya, tkk, bori, bnayenge, banavo, esha, kishi, anaaj, apnaya, seth, thro, koshi, feild,
tuh, apan, ghambir, nikaalte, duniyan, dkhne, samjhoge, moyo, rhai, nale, chasma, tbi,
indai, jitwa, khail, chahiyen, nazaro, gili, esy, toota, kere, ganday, jeta, haga, rae,
jabb, liyea, hamm, abbe, lota, sadko, khulasa, smja, nazro, il, govind, saazish, chlate,
chur, jia, detta, cheen, gandhiji, kila, mazdoor, sunao, sadke, daldal, pado, rukega,
nobat, mahangai, khullam, khulla, fela, nirnay, taali, tumpe, badta, dekhen, tuse, melna,
kiii, mille, khaate, waaley, yeto, woha, ushe, huii, gambheer, wja, isilye, laathi, naata,
sambhalega, nikalega, dhul, jhelo, padte, vinti, bnti, lool, nikalti, koh, brbaad, baahar,
soya, chamatkar, barr, tutna, topa, sarr, goutam, rakkha, ageya, lambe, khoti, majdur, nal,
dehli, dhakel, ann, samajna, srkar, janaab, rakhein, sima, tapasya, kamayi, dio, lejata,
aasaan, chunao, jf, sei, sakun, lathi, chalay, huay, fuzool, kehty, ska, kachara, tumhary,
gaddhe, plx, mandhir, daskhina, thnku, hakk, avidence, kmi, kishano, deshwasiyo, jigra,
niyu, taake, kiti, juli, jayen, munda, kudi, kitti, jeshe, sudharna, jimedari, gayo, lote,
talwa, taan, naheen, sapot, suvida, deas, chamkane, upa, khelke, goray, peele, chhodi,
rizwan, shaheen, sehwag, shahab, ahemdabad, khamosh, shart, mrte, batted, budda, saki,
dosro, injam, maraye, gyaani, chokli, samjhdar, aisee, khaas, lagege, abar, lagni, undar,
khoshi, bachte, okaat, sakhte, bhudape, dooo, haraa, bajate, bhent, peeti, jhukane,
chahiyee, khulengi, kuku, jhak, farmars, peacful, amm, waad, shaab, huai, bahane, vhan,
kavel, khanna, chahiyea, samaan, suji, subse, sunney, illiteracy, thook, sbne, bidesh,
kathin, cmnt, dikhake, danka, chakker, pehda, krda, joga, anaj, kisaaan, roopi, bhagy,
buraa, sosan, isss, bhago, shath, tutti, zubaan, taala, rishab, harna, tona, kahiye, bhuvi,
wajood, layek, khal, jitkar, iyar, khauf, jikr, bating, peechhe, gehra, ghaav, poochho,
hatakar, coch, quki, shreya, bdha, gauti, lndia, boldo, deh, leh, khd, haag, karge, rahahe,
harenge, mese, utrenge, prta, phil, dsp, jitao, samajhna, manmani, badalte, katenge,
kaptani, wapsi, khelao, hena, cheee, manuwadi, zandu, khelunga, wahe, veera, sohna, tuhade,
nri, vari, nall, punjabis, soilders, chukka, cheye, tey, panjabi, loga, rya, aarahe,
acchha, waapas, smne, becharo, ekata, gddar, beguna, virod, samarthak, sharmnak, mangni,
kasur, nanhi, smjhna, jutt, maango, seekhe, pheku, sabun, pajeet, basti, bazti, fekta,
unho, laato, bom, nhee, kaheen, bejti, bhaye, tumhain, bnany, necha, kamzoor, zalil, janty,
dakhi, balky, bholy, waisay, mandira, muskura, unhine, bhoolne, haj, meli, usna, baukhla,
sudh, muskurate, chlte, sakata, dhadkan, jitegi, samajta, khna, batati, athe, ltr,
karneka, sunaya, besharmo, logh, yaari, dilse, phirbhi, dhekho, dikhayegi, batadu, jete,
mukabla, gark, mayoos, rp, rabet, dba, phodo, jabby, bandon, wk, pach, tamiz, badabadia,
naraka, sikhati, deepaavali, bec, deusure, idli, ayesa, enga, vachan, kola, nathi, kirtan,
karthik, vanvaas, atithi, devo, kutumbakam, navi, sia, laaya, tyari, santana, garu, baga,
dhamaka, chinni, ore, viday, badua, pujo, manani, ghayal, daridra, siyaram, urr, gober,
jalai, manaate, gond, sama, mohol, barud, sikhate, khone, chhat, jaaungi, saame, jismein,
taare, jabi, dkha, badhkar, chhodke, sachmuch, paraye, sakhta, boldiya, muchhhh, kashi,
batake, chida, nahh, adaab, lehja, larti, paresani, gaar, chotte, pdegi, tujse, hamaara,
jindegi, chorne, pasnd, paristhiti, swayam, sko, lagey, naho, khabi, bandhan, sharukh, hod,
roki, baale, alwa, dhere, ketne, najr, namkin, mitha, sakee, bdo, jte, pehni, lagrhi,
saddi, mahino, poonam, hanji, kmane, rhae, krdena, ronaa, dane, bona, chak, nangis,
dhamaal, aanso, muchh, kahaani, kaanp, ahsas, dediye, ajaye, ilava, duwa, barfi, subash,
thukrane, dikhaenge, yarrrr, jaggi, firr, betiyan, karav, glat, ahi, akk, vivi, jisay,
ritu, bhabhiji, rabba, daulat, thooo, hasmi, tuch, budiya, budhiya, dhekha, aapn, aape,
kaddu, sikhlo, chaape, kae, barha, darti, mamu, darvaja, daravana, buti, khari, jie, khale,
pagala, tumar, unmen, sota, dari, palak, tota, soti, sabase, boh, meto, rahana, badtameez,
kauwa, darenge, chhaya, dugi, samee, lagayi, gyii, bakte, videsho, lgani, pawan, soda,
jagte, uthte, mahakaal, preshn, pilao, pur, mochan, kono, fitkari, sthapit, baandh,
ghoomti, padosiyon, padao, chok, buch, maroo, nayab, feke, tariqa, dosre, yesab, birodh,
gharib, ishare, sabot, apraadhi, apradh, bhay, darinde, surakshit, royega, shivay,
chalawa, chalava, chhalava, dhoond, bnani, khraab, anurodh, feko, fekuchand, mukhiya,
kirpa, khaaya, simit, launga, nikaalne, aba, laka, peete, udane, cheezon, niklna, niii,
dande, unlog, mahilaon, jallad, paisaaa, sekte, zabrdasti, shaddi, hunn, aahista, ahista,
btaun, ghani, namaskaar, shehar, logoka, mardi, vahin, lgana, gunehgar, rihaa, maala,
zaher, deshvasiyon, sanshodhan, dhaan, veerta, thokna, ittt, deewar, shabdo, wande, bnaungi,
jahir, jalta, royi, mujha, khash, majaa, bharata, dhunde, huhuhu, una, zda, hojao, ufffff,
kheir, puchhe, kamiyabi, saan, akh, tmhre, intjaar, apnee, kesari, gle, kapdo, kasa, ghatak,
banaane, aagayi, tuu, pn, lgrha, pasia, eesa, kamo, seene, pdne, mehra, kav, aasun, parhi,
jbb, dauran, manam, tese, shivaya, devuda, valla, hamere, abaj, gawaar, kdi, jaani, umr,
dayar, kos, admiyo, ubal, chihye, todhi, jism, dhiyan, ehsas, chayea, khasa, botal,
dikhaungi, manuga, deo, ras, las, jygi, bado, bachapan, amiro, vadi, hatavo, gandh,
khilor, hava, awara, sparsh, harny, dhaar, amna, chahiay, ahaa, gunga, muse, bachoun, thei,
logi, janaza, pory, jageer, sda, bhram, qk, bhartye, dhoom, daalna, chodkr, karwaye,
tambaku, banaate, munafiqat, chillake, avam, buttni, apku, ima, payr, chaploosi,
bharatiye, bandariya, nathu, manik, sambhali, chhodne, bela, ghantal, akheer, suruvat,
mantriyo, gaate, jj, samadhi, nazriya, washio, assalaam, samandar, waqar, raaste, doosro,
ngk, thikane, khandaan, awo, hoker, jann, bapauti, lrki, pakista, kartavy, jeeth, jasy,
mananiy, karyo, chahty, ummide, meer, atankwad, aser, bachiyo, bdhai, qaum, sahafi, ganjay,
lashein, bichh, pohanch, bharwi, kitaabe, taptan, sallu, aisey, jaisey, paidawar, kuon,
wahid, tumhein, neeti, kamna, aukaad, pardhan, mantari, aq, bhavisya, kirti, gribo, tmhe,
hram, gulshan, sunnat, dekhai, jeeyo, noch, sawalon, jaanti, insaafi, dukhta, kittu, katt,
chaplus, ahankar, royenge, owa, khalistaan, harrami, mangna, ulad, tiranga, daa, naina,
doba, melay, yadein, gadhar, dehk, nik, chewtiya, aapako, bistar, sktay, bnege, mantr,
jabrdasti, bachu, dilana, olaad, hoor, batti, unake, dkhte, naik, karvaye, kol, neya, kast,
manata, kabar, jagi, beiman, karaa, maarke, dilwao, aako, pagalpanti, garima, berojgaro,
bayaj, wasooli, bharwe, chy, jumle, pankh, duni, aanchal, tughe, dou, jhansi, padhata,
udher, kandhe, chamak, kuran, ghana, aashirvad, sarangi, baaat, sadaiv, mily, aakhiri,
lootero, lauda, naxal, khella, indrani, deewana, alikum, lak, chhoot, jyegi, ishara,
puchiye, angrezon, bnni, parantu, bhajji, barsat, mithun, humri, rushna, enke, paran, aho,
taka, chillane, bah, eis, junoon, lasho, badalta, karkare, krtay, aatankvadi, thoo, dhora,
bhagana, hashi, lahu, dhakkan, kak, nadi, wafadar, mander, sbr, jaogay, bhatiya, khon,
cheekh, mzak, rajneti, haai, jehar, fiada, naate, daina, daliye, shahido, pakistaan, lgati,
jagaa, sambhavna, jodha, khushkhabri, londo, ekbaar, maira, parbo, aktu, pasha, kichu,
gujarish, sadhi, sainiko, susti, aun, dilata, sidarth, niye, shiddharth, hojate,
shradhanjali, sahido, jih, korne, sunlo, jubin, rata, mitt, arabpati, munafa, nojawan,
ashaji, puraskar, chhip, bhavpurna, atankvaadi, kithe, pehchaan, dikhava, jimedar, bolei,
dekhechen, kathputli, istefa, deemak, lallantop, rehkar, dallal, jawanoon, vatan, jahad,
chaihe, kaushik, waar, gaadron, haramkhur, gadgari, hatawo, dalenge, jori, tadipaar,
khilap, krusi, ramu, trung, hayega, hindutwa, shambhala, hai, ko, ka, se, bhi, ho, bhai, nahi, ye, aur, nhi, kar, kya, jo, ne, ji, ek, tha, na, koi,
log, aap, desh, hain, sab, ab, liye, toh, kuch, bahut, raha, mein, mai, pe, par, diya, tum,
wo, rahe, kr, jai, apne, chor, ghar, kiya, baat, hu, hota, gaya, tu, yeh, mere, mandir,
chahiye, har, mujhe, de, aaj, kisi, hoga, rha, hua, khud, thi, jab, karo, ya, aa, hum, sath,
bhaiya, itna, wale, mera, kyu, sahi, sirf, tak, bol, lekin, rhe, maa, naam, kam, meri,
bhagwan, karne, pata, sarkar, apni, karte, chori, nahin, le, fir, dil, aise, rahi, abhi,
wala, jaise, di, vo, paisa, mat, aisa, acha, apna, kaam, gaye, bana, hoti, karna, hamare,
liya, janta, hote, kon, iss, baap, ja, bohot, bas, baar, tere, sach, yaar, sabse, laga,
zindabad, paise, aapko, teri, jata, lagta, kaha, dete, phir, lo, saal, nai, karta, itni,
ham, yahi, kitna, kaise, mar, dena, hona, didi, pura, bola, jyada, shree, rhi, bada, lag,
bhartiya, tab, bhut, uske, sakta, tarah, baba, khush, galat, hone, aapki, woh, jeet,
kuchh, kha, tera, gye, karke, kro, kal, yah, hun, chal, saath, yaha, krte, usko, unko, sare,
mei, maine, sakte, aapke, ekta, jaye, pta, gayi, wali, apko, kare, hui, dono, loot, logon,
unke, duniya, aapne, dekha, dekho, tho, kharab, itne, neta, jis, aapka, mil, krne, isliye,
hue, banaya, accha, bina, inko, puri, sala, apka, aaya, sabhi, deta, aata, bhot, chutiya,
gyi, bs, nikal, bahar, dadi, bolo, hogi, khana, cute(removed), shri, chala, khushi, pagal,
dharm, jayega, bolte, apke, hamesha, bade, jao, krta, gai, kia, bilkul, jaan, bare, soch,
abe, achha, yaad, isko, upar, mila, krna, sun, aage, kyuki, uska, wah, kitne, insan, bna,
matlab, diye, dekhne, saale, jb, mast, godi, kab, ghr, jate, maar, dene, aati, aj, beta,
jagah, apki, bahot, iska, badi, unki, bura, aya, khatam, iske, tumhare, dhanda, uski,
achi, insaan, kaun, kyun, ladki, baki, chale, jaldi, naa, nehi, inke, isse, maza, hamara,
aaye, eakta, aisi, mehnat, sabko, maja, thoda, denge, bi, leke, pyar, hoon, bata, jaisa,
hame, chutiye, jindabad, dekhna, kahi, ese, kutte, sari, banao, kis, agr, sal, karega,
wahi, banane, inki, galti, mile, vah, chaiye, bhar, bht, bacha, bolne, aadmi, aam, hamari,
rona, mene, bolna, lena, bhoot, kitni, chod, dukh, badhai, mata, yar, isi, dikha, waha,
dia, thik, lage, samne, ata, lga, unka, ache, bhakt, dharam, bole, acche, jati, alag, jane,
mara, piche, khane, samajh, usse, raho, karti, kasam, sara, kyunki, karenge, dusre, es,
acchi, kyo, kiye, jitna, zindagi, dhan, haar, chanda, jana, muh, hath, ky, shi, hm, chahta,
hindi, tumhe, vlog, tujhe, tumhari, jaa, saare, andar, pasand, sahab, ker, bakwas, inka,
kese, banda, honi, chahie, kyon, hindus, achcha, beti, dega, pani, yahan, wajah, samaj,
garib, jese, hume, humare, asli, iski, tune, usne, isme, gand, admi, jada, badhiya, dost,
chup, gali, tabhi, jaha, milta, lgta, chuka, muje, zyada, dimag, jarurat, marne, himmat,
haal, chahe, crore, leta, pa, lekar, radhe, shanti, hatao, gareeb, sharam, masjid, lagi,
jitne, gussa, taraf, halat, tumhara, barbad, bohat, hindustan, nazar, jayenge, tumne,
milega, esa, jhoot, dunga, dal, puja, jayegi, bache, saja, dekhte, hoo, chalta, jaate,
aayega, ro, jaaye, sabka, evm, aik, faltu, tumko, padega, harami, mt, banate, sakti,
thodi, hei, salo, maro, krke, khel, kohli, theek, dekhe, jarur, andhbhakt, jee, jaane,
samjh, kafi, rakh, dalal, daan, raat, padta, gyan, tou, deti, isiliye, roti, ku, abb, yhi,
ge, sake, pyaar, raam, mc, phle, ati, yad, mtlb, krishna, waqt, choro, dikh, achhe, aye,
dard, karan, waise, sapna, bech, chahte, jinda, mili, keh, dua, dar, dur, shuru, yha,
hanuman, kutta, khilaf, kash, gaali, krti, mare, bhaii, rai, gaand, sarkaar, usi, mantri,
behen, chuki, jawan, chalega, aukat, tb, tm, waah, ch, lagti, banaye, badal, bss, shayad,
banne, bta, aana, atleast, lg, skte, dekhkar, zinda, chowkidar, unhone, rah, gi, sbse,
bhe, bhul, asha, baare, dekhta, han, bahi, fr, rakhe, hmare, aab, nafrat, naya, khusi,
taki, milna, gandi, jaati, batao, musalman, ayega, huye, chote, chalo, fayda, haram, sewa,
aapse, kehte, bande, inhe, layak, likha, daal, paida, chuke, lanat, madarchod, sai, lenge,
dhandha, khate, jin, gaddar, dan, saaf, shadi, khol, bnaya, ladke, chhod, bn, sabke, sawal,
bharosa, marte, socha, jake, dekhi, jhut, sikh, gadi, barbaad, maut, lagte, aulad, chiz,
hr, saalo, saab, usme, lia, mann, rahenge, bohut, ghanta, jhooth, esi, kamal, tod, chu,
jan, kanoon, rehta, gay, vale, hogaya, bo, boht, izzat, jawab, banta, pade, nikla, mahan,
buri, kami, hogya, fasi, dosto, utha, mumkin, khoon, kah, marna, vikas, achhi, saza, mulk,
chali, bolti, milti, kaa, baith, khus, hal, bap, chakkar, siya, rat, badiya, bhagwaan,
bnao, andh, banaa, bacho, jaat, isne, dalo, niche, baithe, bik, paiso, thaa, lakin, dii,
umar, bachpan, padh, jaisi, rakhna, jisko, khatm, kaisa, koshish, rula, kisne, bakchodi,
hisab, pesa, lao, maan, vande, chunav, lega, bda, awaaz, awaz, dushman, pese, kch, saala,
aacha, mana, jal, payega, bane, janata, jii, gar, aagya, poora, jesa, haath, bkl, bolega,
badh, karoge, vai, andhbhakto, haa, haan, roj, vaise, sat, aag, yarr, chij, ajj, chahti,
mahine, socho, aake, paya, bhej, sarkari, shaadi, jindagi, jisme, bete, mn, chudail, gaadi,
dhongi, pehli, bataya, baaki, karen, uper, choti, babu, wohi, kush, varna, bharatiya,
haii, pareshan, bass, mushkil, dusro, saari, baate, bhakto, sona, waale, bh, khuda, fansi,
smjh, baatein, karwa, kardo, umeed, dusri, bhaut, bich, glt, jiske, chai, banna, khali,
jit, kisano, unse, hoge, jitni, jala, khata, ameen, dhyan, bandi, phr, milegi, bachi,
dekhti, zameen, humara, gae, dala, ratna, sidha, bhen, kahe, roshan, hmesha, pakad, milne,
bhakti, zindabaad, banega, poori, ghotala, akele, badla, dikhaya, majak, bal, tumse,
aasra, ayodhya, likh, gud, amir, bolenge, dusra, keliye, bahan, apse, ganda, kari, bach,
bhaiyo, pyari, aurat, badnam, gandu, humko, bara, humari, kaho, chota, jara, hasi,
bachane, bhala, voh, wahan, jahan, dikhao, dukan, jinke, rehne, khilaaf, pati, deni,
tatti, deke, nani, kaafi, gunda, veer, kisaan, aansu, kehna, uth, kiu, kijiye, padi,
banata, netao, ate, hat, ti, badnaam, alawa, jinko, bnane, roz, bacche, murkh, madad,
sunta, musalmano, garibo, ghus, karunga, hii, gayab, lge, vdo, yaa, humne, hae, dikkat,
sunte, farak, kat, padhai, jaoge, du, rajya, farzi, choda, gulam, khao, galt, kyoki,
rehna, rehe, mu, abey, karegi, baitha, banai, rahy, deshdrohi, rahte, kaat, uspe, bhag,
khtm, khul, dher, bari, vajah, rakhi, ane, aese, haste, bechare, aatma, vahi, rok, sbko,
pahli, huwa, bhool, mae, bandh, vishwas, ganji, chalu, lana, kama, bate, aqi, pai, khaya,
ramadan, samjha, pichle, dikhane, milawat, insaaf, khelne, tarike, mey, vaha, krenge,
jaruri, saamne, nikalo, des, bde, mudda, khub, krega, mante, jisse, ghum, kalyug, mudde,
sunne, nu, boli, thee, rehti, jiska, jaga, rhy, mazak, ghante, koe, idhar, rahta, nautanki,
gy, oye, lagega, humesha, nakli, ladka, milkar, paa, leti, phele, bajrangbali, jodi,
mama, bolu, lok, lut, jha, banakar, naukri, jinhone, sunke, kl, walay, pada, dikhta, kra,
trah, khila, kahte, bhagat, ameer, hamre, tukde, atma, dino, ladkiyon, chla, inshallah,
sabki, ladkiyo, koti, gir, kehta, jeete, abi, tarha, wapis, kabi, bhara, mi, agle, keya,
maaf, aesa, jitega, baccho, bhartiye, dhoka, kaash, marunga, padha, yh, vala, sapne,
shakal, larki, moka, mamta, aankh, khi, aai, dikhana, sochte, pucho, rajneeti, sadak,
ijjat, kyaa, tuje, chaye, laat, kho, apny, baccha, agya, gajab, karu, videsh, bhee, nhe,
bai, batane, chl, bhari, sher, hara, rote, hadd, zada, doge, pel, bhau, daba, krdo, nikle,
jamin, hamesa, krr, yuva, pyara, rd, bhaga, behan, aagaya, hogye, joh, andha, jyda,
aukaat, loota, aastha, haq, deshbhakt, skti, kharid, jeeta, thak, jaake, marta, jeetne,
aadat, gobar, pehla, zarurat, dalali, jae, prem, anpadh, chamche, hoke, ander, hokar,
sanatan, wa, rone, mja, kise, naye, namak, zaroorat, dekar, malik, jankari, gaon, maje,
waja, deya, raaj, kyonki, malum, thode, utar, seedha, lgi, milke, inhi, ri, paap, der,
pichhe, karlo, konsa, mari, kee, bethe, rajniti, akal, deepavali, likhe, hmara, hin,
bhiya, rkha, chalte, darr, khaa, taaki, khabar, lye, pakhandi, hinduon, sumiran, jumma,
kishan, khrab, mang, sunna, cha, krwa, jakar, rang, nikala, jiski, lgti, kisiko, maaro,
sambhal, chuna, hamne, jahil, aapane, krty, badha, alg, yrrr, gale, valo, aasu, pi, purani,
batana, dare, lambi, akhir, faida, laya, dhokha, aayenge, dimaag, chhor, sikha, wese,
ispe, bikau, bahu, aapas, lad, jise, poore, muskil, dunia, mane, liy, beth, rhega,
afghani, bhart, lakshmi, jaby, huaa, dedo, janam, vhi, rhte, rahne, lootne, dwara, dhamki,
bhagwa, suar, dunya, nii, chaina, nanga, kissan, sachi, andolan, fer, paani, thora,
ghamand, aajkal, dev, virodh, gira, chodo, dosti, achara, rahata, bacchon, dijiye,
rakhta, sochta, sanam, biwi, awaj, jald, thappad, kadi, jeb, kaand, dharmik, sch, hmari,
ayi, jita, ine, pakda, bnd, chiye, hoa, haiii, banti, khada, bne, jesi, udhar, kala, sone,
dukhi, manta, kbi, jameen, darshan, kardiya, bhaiiii, ae, masoom, insaf, lajawab, ghatna,
nae, vali, lagana, padhe, chora, issi, himat, lakho, hawa, jayda, aulaad, jaana, pate,
saat, aor, lagao, ummeed, aakhir, kewal, lagata, jeeto, umid, gadhe, hari, imandar,
lagate, asal, chalti, sayad, premiyo, mujhse, banayi, bure, ullu, sunkar, maha, kapde,
hamen, tai, hissa, ammi, kariye, jaunga, chapri, mkc, kursi, chut, btao, chilla,
chokidar, bhenchod, fauj, sohne, geya, afsos, bhole, mota, khule, ana, bhakts, purana,
behtar, tilak, bhasha, khas, lekr, bimari, bagwan, dudh, rahoge, pari, ukhad, khair,
jayada, chamcha, shamil, mereko, dek, zara, kesi, bheja, dekhke, roop, ami, bhaag,
karein, haramkhor, choor, loge, uthate, babao, mullo, dalit, babri, shifu, vyakti, bnda,
nhii, khaye, pahele, bechne, koyi, ansu, chata, degi, galiya, huyi, ismein, santi, rasta,
kavi, nayi, lata, uthaya, bhaiyaa, azadi, btaya, nahe, nana, ankho, jannat, ladai,
samjhe, mashaallah, karwaya, lagne, karva, bevkuf, vahan, ldki, jhutha, howa, aani,
khun, ani, gawar, thok, mataram, batein, hatya, molvi, chaudhary, rahay, gazab, nuksan,
lagaye, khelte, unhi, agli, agla, karege, kh, lal, chadha, jite, sey, guruji, pranam,
mst, aree, bahat, jaao, chhote, gujarati, shukriya, bhaiyon, radha, hindustani, chura,
garba, das, suwar, andr, bnate, tbhi, mee, naseeb, rahegi, padhne, ladkiya, aayegi,
chehra, bachhe, pariwar, ghee, abba, marti, ajeeb, kilhor, saheb, raghav, angrejo,
harkat, kaya, ramji, aasha, behn, sorav, bhosdi, chutia, bachy, satya, humein, chatne,
bewakuf, gulami, jaty, tumlog, odi, tara, namo, aatankwadi, baji, sadi, bhav, trh, haat,
bak, hindutva, manmohan, vapas, fayde, munde, ghalat, banati, atankwadi, dalna, qki,
najar, abt, utne, fek, juda, hongi, shukar, pehele, jeena, dikhai, gham, rishta, abh,
jiye, gande, faad, hamko, akela, andhe, mahila, aayu, turant, koun, gala, dogle,
chaukidar, maat, kamai, ujjain, dhurandhar, kahenge, jhuth, likhi, banake, dill, danda,
wha, gane, astha, lode, gaddaro, dhanyawad, sacha, aawaj, baten, unn, congres, leye,
arre, shreyas, naak, harne, kesa, takleef, kharcha, mandi, badle, saf, chalate, meine,
batau, keval, krny, nasha, yani, anna, bihari, chi, backchodi, thu, dekhenge, kii, bato,
mano, tumara, khareed, bhare, jeevan, tujh, ladko, jindgi, achche, safar, masha, bisht,
subha, dhire, bagal, ruk, batate, ghor, vishwaguru, hogaye, avi, thoko, aaisa, kri,
dekhunga, khor, insaniyat, tukaram, choron, bologe, chuha, khaane, paar, kaudi, jor,
bhad, kodi, bahen, sasta, drohi, bhadwe, sanyas, sunti, bahoot, aadami, druv, hora,
ekdam, jeetega, shami, bhejo, bolunga, hazar, dalle, lunga, achaa, lay, agaya, bahir,
laa, dukaan, jihad, karate, kaisi, verma, inlogo, kiska, dila, tareef, besharam, dogla,
rahti, bdi, rakhne, anpad, pee, dusron, rashtra, baato, adat, aapna, firse, bigad,
daru, pagla, juth, maarne, sita, paji, hansi, karaya, swagat, tagda, padosi, adhuri,
aadhe, rota, janm, manga, rahna, raksha, jldi, dobara, puch, chlega, pit, hass, moot,
lala, konse, chachi, pol, huva, pujari, raste, tumari, imandari, ake, jalan, bhii,
chhoda, sanskar, sree, lai, apnay, jaenge, vipaksh, dharmendra, dhirendra, bulaya,
visvas, malhan, bhaijaan, uthane, rahiye, chun, hoi, kiyu, gyaan, doo, dho, loktantra,
tumare, jagha, chillate, waala, asan, aasan, apane, khiladi, kosis, pela, glti, hmesa,
upr, kehne, pda, kita, bharose, istifa, kerte, cahiye, nei, whi, holi, bhikhari, daka,
banaiye, yatra, gaana, hoty, yakin, lac, aesi, uthana, janwar, pdega, sikhaya, maang,
saboot, shakti, milni, sudhrega, vha, bewkuf, honga, kamyabi, muze, paaye, sbhi, kiske,
kitno, jimmedar, loog, changa, chalne, fal, gareebo, saccha, gaa, horaha, dheere, suraj,
babas, sunn, waqf, jawano, andhbhakti, dali, harsh, bachho, salaam, sacchi, pucha,
bhakton, ajadi, mahakal, chalegi, jaante, phat, jaanch, hogyi, saka, kata, zaroor,
sochna, ungli, lelo, krni, kerna, padegi, khadi, jat, khalistan, nikalna, yakeen, bale,
khuch, banayenge, sanatani, jayege, karn, hoye, pradhanmantri, shaan, bhukhe, ohhh,
muddo, cheezo, ishwar, niyat, kuj, aae, ulti, yai, bakwaas, samjhte, padenge, dalne,
bataye, paoge, baja, laxmi, tihar, samajhte, uda, usmein, bla, safai, lgte, hazaro,
haye, bechari, samajhne, kahta, baal, hisaab, bibi, kry, salamat, krege, tarif, ghajini,
kamana, desi, kholo, nakali, tume, aakar, diyaa, appko, nyay, hajaro, intezar, ilzam,
sabi, mughe, dikhti, lu, josh, kholi, jabtak, nuksaan, saara, jali, patti, doodh, farq,
laal, jaag, marr, luta, bnata, azam, madat, mitti, muslimo, mulle, namaskar, ramzan,
hak, pde, samjho, pakdo, sukh, khoob, bekaar, khilao, apno, hamaare, jinka, samja,
apman, patthar, fasal, jub, mafi, chatukar, mazhab, khelta, chalana, esse, parega,
kamaal, chah, zaleel, chhota, bhagao, chupa, deepawali, mahol, jaayega, boliye, tau,
masla, marzi, betha, etne, mamla, jago, khubsurat, rupaye, padho, shubh, kuda, istemal,
sabh, jaya, meko, yaadein, purane, nasib, ashirwad, baje, humesa, huu, mujy, tarakki,
dekhiye, achchi, ley, bnana, fauji, sbke, akshay, duaa, kher, tarf, mohabbat, dede,
dara, padhte, moti, kute, payal, kissi, farji, rhta, chord, bnai, maloom, kidhar, macha,
chahye, trha, aankhe, batata, chlta, matlb, nikaal, khula, bhgwan, bachon, kattar,
kroge, neeche, shiksha, uthao, andhi, kiun, banegi, parchi, dhram, sidhu, zarur, madar,
gayee, madarchodo, ibadat, shab, akta, sachai, sacchai, ankhe, taklif, jivan, aazad,
khele, doob, ay, khelo, waali, vese, dekhni, uttar, thuk, hrami, krlo, bnaye, janti,
kahna, chate, inme, hosh, lagatar, dosh, aare, payi, kaptan, lagakar, latka, waisa,
daar, panga, murdabad, samj, plss, samna, mamle, hans, thekedar, sahib, smj, jine,
kiss, andhbhakts, ghumne, mazza, paate, aaplog, afgan, pyare, patakhe, khani, prathna,
subh, hmlog, pathar, peet, pakhand, ske, daadi, khyal, jaroor, mumma, unpe, mea, aaap,
dungi, den, rhna, nikalta, zaara, chle, laate, chalisa, mantar, bageshwar, munh,
barish, dhak, hotay, aarop, bacchi, samil, kas, zor, ganja, laane, bimar, mout,
ladkiyan, berojgar, jiya, rhne, jyadatar, rhaa, chhodo, gutka, var, chhoti, chappal,
mna, dikhega, krdiya, bapu, khaunga



"""